In [1]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
import contextily as ctx
import warnings
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from scipy.stats import entropy as shannon_entropy

warnings.filterwarnings('ignore')

print("All imports successful.")
print(f"OSMnx version: {ox.__version__}")

All imports successful.
OSMnx version: 2.1.1


In [2]:
# ── RERUN LONDON THROUGH STANDARD PIPELINE ────────────────

london_standard = run_pipeline(
    city_name="London",
    streets_file="/home/jovyan/work/data/london_streets.gpkg",
    buildings_file="/home/jovyan/work/data/london_buildings.gpkg",
    pois_file="/home/jovyan/work/data/london_pois.gpkg",
    epsg=27700
)

print(f"London reprocessed: {len(london_standard)} segments")
print(f"Columns: {list(london_standard.columns)}")

NameError: name 'run_pipeline' is not defined

In [ ]:
# ── CLUSTER LONDON USING STANDARD PIPELINE ────────────────

feature_cols = [
    'length_m', 'building_count_50m', 'avg_building_area',
    'poi_count_50m', 'highway_rank', 'sinuosity',
    'poi_diversity', 'food_drink_count',
    'street_furniture_count', 'avg_connectivity'
]

london_standard_clustered, lon_scaler, lon_kmeans = cluster_city(
    london_standard.reset_index(), "London")

In [ ]:
# ── NAME LONDON CLUSTERS CONSISTENTLY ─────────────────────

london_name_map_standard = {
    0: 'Active Commercial High Street',
    1: 'Institutional Large-Block Street',
    2: 'Local Mixed Street',
    3: 'Winding Historic Lane',
    4: 'Pedestrian-Rich Street'
}

london_standard_clustered['typology'] = london_standard_clustered[
    'cluster'].map(london_name_map_standard)
london_standard_clustered['city'] = 'London'

print("London typologies:")
print(london_standard_clustered['typology'].value_counts())

In [ ]:
# ── UPDATE ALL CITY DATAFRAMES TO USE STANDARD LONDON ──────

# Rebuild combined database with standardised London
all_streets_updated = pd.concat([
    london_standard_clustered[feature_cols + ['typology', 'city', 'name']],
    barcelona_clustered.reset_index()[feature_cols + ['typology', 'city', 'name']],
    singapore_clustered.reset_index()[feature_cols + ['typology', 'city', 'name']],
    tokyo_clustered.reset_index()[feature_cols + ['typology', 'city', 'name']]
], ignore_index=True)

all_streets_updated['name'] = all_streets_updated['name'].fillna('Unnamed Street')

def clean_name(name):
    s = str(name)
    if s.startswith('['):
        return s.strip("[]'\"").split("',")[0].strip("' ")
    return name
all_streets_updated['name'] = all_streets_updated['name'].apply(clean_name)

print(f"Updated database: {len(all_streets_updated)} streets")
print(f"\nCity breakdown:")
print(all_streets_updated['city'].value_counts())
print(f"\nTypology breakdown across all cities:")
print(all_streets_updated['typology'].value_counts())

In [ ]:
# ── UPDATED CROSS-CITY COMPARISON TABLE ───────────────────

city_dfs_updated = {
    'London': london_standard_clustered,
    'Barcelona': barcelona_clustered.reset_index(),
    'Singapore': singapore_clustered.reset_index(),
    'Tokyo': tokyo_clustered.reset_index()
}

summary = []
for city, df in city_dfs_updated.items():
    total = len(df)
    counts = df['typology'].value_counts()
    for typology, count in counts.items():
        summary.append({
            'City': city,
            'Typology': typology,
            'Count': count,
            'Percentage': round(count / total * 100, 1)
        })

summary_df = pd.DataFrame(summary)
pivot = summary_df.pivot_table(
    index='Typology',
    columns='City',
    values='Percentage',
    aggfunc='sum'
).fillna(0).round(1)

print("Updated cross-city typology distribution (% of segments):")
print(pivot)

pivot.to_csv(
    "/home/jovyan/work/data/cross_city_comparison_final_v2.csv")
print("\nSaved.")

In [ ]:
# ── FULL PIPELINE FUNCTION ─────────────────────────────────
# This function takes a city name and its saved data files
# and returns a GeoDataFrame with all 11 features and typology labels

def run_pipeline(city_name, streets_file, buildings_file, pois_file, epsg):
    print(f"\n{'='*50}")
    print(f"Running pipeline for {city_name}...")
    print(f"{'='*50}")

    # ── Load saved data ────────────────────────────────────
    edges = gpd.read_file(streets_file)
    buildings = gpd.read_file(buildings_file)
    pois = gpd.read_file(pois_file)
    print(f"Loaded: {len(edges)} streets, {len(buildings)} buildings, {len(pois)} POIs")

    # ── Project to metres ──────────────────────────────────
    edges_proj = edges.to_crs(epsg=epsg)
    buildings_proj = buildings.to_crs(epsg=epsg)
    pois_proj = pois.to_crs(epsg=epsg)

    # ── 50m buffer ─────────────────────────────────────────
    edges_buffered = edges_proj.copy()
    edges_buffered['geometry'] = edges_proj.geometry.buffer(50)

    # ── Feature 1: length ──────────────────────────────────
    edges_proj['length_m'] = edges_proj.geometry.length

    # ── Feature 2: building count ──────────────────────────
    joined_b = gpd.sjoin(edges_buffered[['geometry']],
                         buildings_proj[['geometry']],
                         how='left', predicate='intersects')
    edges_proj['building_count_50m'] = joined_b.groupby(
        joined_b.index).size().reindex(edges_proj.index, fill_value=0)

    # ── Feature 3: avg building area ───────────────────────
    buildings_proj['area'] = buildings_proj.geometry.area
    joined_ba = gpd.sjoin(edges_buffered[['geometry']],
                          buildings_proj[['geometry', 'area']],
                          how='left', predicate='intersects')
    edges_proj['avg_building_area'] = joined_ba.groupby(
        joined_ba.index)['area'].mean().reindex(edges_proj.index, fill_value=0)

    # ── Feature 4: POI count ───────────────────────────────
    joined_p = gpd.sjoin(edges_buffered[['geometry']],
                         pois_proj[['geometry']],
                         how='left', predicate='intersects')
    edges_proj['poi_count_50m'] = joined_p.groupby(
        joined_p.index).size().reindex(edges_proj.index, fill_value=0)

    # ── Feature 5: highway rank ────────────────────────────
    def clean_highway(val):
        if isinstance(val, list):
            return val[0]
        return val

    highway_map = {
        'trunk': 6, 'primary': 5, 'primary_link': 5,
        'secondary': 4, 'secondary_link': 4,
        'tertiary': 3, 'tertiary_link': 3,
        'residential': 2, 'living_street': 2, 'unclassified': 1
    }
    edges_proj['highway_clean'] = edges_proj['highway'].apply(clean_highway)
    edges_proj['highway_rank'] = edges_proj['highway_clean'].map(
        highway_map).fillna(1)

    # ── Feature 6: sinuosity ───────────────────────────────
    def calculate_sinuosity(geom):
        if geom.length == 0:
            return 1.0
        start = geom.coords[0]
        end = geom.coords[-1]
        straight = ((end[0]-start[0])**2 + (end[1]-start[1])**2)**0.5
        if straight == 0:
            return 1.0
        return geom.length / straight

    edges_proj['sinuosity'] = edges_proj.geometry.apply(calculate_sinuosity)

    # ── POI categories ─────────────────────────────────────
    food_drink = ['restaurant', 'fast_food', 'cafe', 'pub', 'bar']
    street_furniture = ['bench', 'fountain', 'clock', 'telephone', 'post_box']

    def categorise_poi(row):
        amenity = str(row.get('amenity', ''))
        shop = str(row.get('shop', ''))
        if amenity in food_drink:
            return 'food_drink'
        elif amenity in ['bicycle_parking', 'bicycle_rental',
                         'motorcycle_parking', 'parking']:
            return 'transport'
        elif amenity in ['bank', 'atm']:
            return 'finance'
        elif amenity in ['place_of_worship', 'doctors',
                         'dentist', 'pharmacy']:
            return 'civic'
        elif amenity in street_furniture:
            return 'street_furniture'
        elif shop not in ['nan', '']:
            return 'retail'
        else:
            return 'other'

    pois_proj['category'] = pois_proj.apply(categorise_poi, axis=1)

    # ── Feature 7: POI diversity ───────────────────────────
    joined_cats = gpd.sjoin(edges_buffered[['geometry']],
                            pois_proj[['geometry', 'category']],
                            how='left', predicate='intersects')

    def calculate_entropy(group):
        counts = group['category'].value_counts()
        return shannon_entropy(counts)

    poi_entropy = joined_cats.groupby(joined_cats.index).apply(calculate_entropy)
    edges_proj['poi_diversity'] = poi_entropy.reindex(
        edges_proj.index, fill_value=0)

    # ── Feature 8: food and drink count ───────────────────
    food_pois = pois_proj[pois_proj['category'] == 'food_drink']
    joined_food = gpd.sjoin(edges_buffered[['geometry']],
                            food_pois[['geometry']],
                            how='left', predicate='intersects')
    edges_proj['food_drink_count'] = joined_food.groupby(
        joined_food.index).size().reindex(edges_proj.index, fill_value=0)

    # ── Feature 9: street furniture count ─────────────────
    furn_pois = pois_proj[pois_proj['category'] == 'street_furniture']
    joined_furn = gpd.sjoin(edges_buffered[['geometry']],
                            furn_pois[['geometry']],
                            how='left', predicate='intersects')
    edges_proj['street_furniture_count'] = joined_furn.groupby(
        joined_furn.index).size().reindex(edges_proj.index, fill_value=0)

    # ── Features 10 and 11: connectivity and betweenness ──
    print(f"Calculating network features for {city_name}...")
    G = ox.graph_from_place(city_name, network_type="drive") if city_name == "City of London, UK" else None

    # Use degree from edges index as connectivity proxy
    edges_reset = edges_proj.reset_index()
    if 'u' in edges_reset.columns and 'v' in edges_reset.columns:
        # Build degree from u and v columns
        all_nodes = pd.concat([edges_reset['u'], edges_reset['v']])
        degree_dict = all_nodes.value_counts().to_dict()
        edges_reset['degree_u'] = edges_reset['u'].map(degree_dict).fillna(2)
        edges_reset['degree_v'] = edges_reset['v'].map(degree_dict).fillna(2)
        edges_reset['avg_connectivity'] = (edges_reset['degree_u'] +
                                           edges_reset['degree_v']) / 2
        edges_proj = edges_reset.set_index(['u', 'v', 'key'])
    else:
        edges_proj['avg_connectivity'] = 4.0

    # ── Add city label ─────────────────────────────────────
    edges_proj['city'] = city_name

    print(f"{city_name} features complete.")
    print(f"Features calculated: length_m, building_count_50m, avg_building_area,")
    print(f"  poi_count_50m, highway_rank, sinuosity, poi_diversity,")
    print(f"  food_drink_count, street_furniture_count, avg_connectivity")

    return edges_proj

In [ ]:
# ── RUN PIPELINE ON ALL THREE CITIES ──────────────────────
# Using the correct EPSG projection for each city
# EPSG 3857 is Web Mercator — works globally

# Barcelona
barcelona = run_pipeline(
    city_name="Barcelona",
    streets_file="/home/jovyan/work/data/barcelona_streets.gpkg",
    buildings_file="/home/jovyan/work/data/barcelona_buildings.gpkg",
    pois_file="/home/jovyan/work/data/barcelona_pois.gpkg",
    epsg=3857
)

# Singapore
singapore = run_pipeline(
    city_name="Singapore",
    streets_file="/home/jovyan/work/data/singapore_streets.gpkg",
    buildings_file="/home/jovyan/work/data/singapore_buildings.gpkg",
    pois_file="/home/jovyan/work/data/singapore_pois.gpkg",
    epsg=3857
)

# Tokyo
tokyo = run_pipeline(
    city_name="Tokyo",
    streets_file="/home/jovyan/work/data/tokyo_streets.gpkg",
    buildings_file="/home/jovyan/work/data/tokyo_buildings.gpkg",
    pois_file="/home/jovyan/work/data/tokyo_pois.gpkg",
    epsg=3857
)

print("\nAll three cities complete.")

In [ ]:
# ── REBUILD SIMILARITY SEARCH WITH UPDATED DATABASE ───────

X_all = all_streets_updated[feature_cols].fillna(0).values
scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

knn = NearestNeighbors(n_neighbors=6, metric='cosine')
knn.fit(X_all_scaled)

print(f"Similarity search rebuilt: {len(all_streets_updated)} streets")
print("Ready to search.")

In [ ]:
# ── LOAD LONDON AND COMBINE ALL FOUR CITIES ───────────────
# Load London final features from the existing pipeline
london = gpd.read_file("/home/jovyan/work/data/london_streets.gpkg")
london = london.to_crs(epsg=3857)
london['city'] = 'London'

# Load London features from the saved CSV
london_features = pd.read_csv("/home/jovyan/work/data/london_final_features.csv")

# Combine all four cities into one dataframe
all_cities = pd.concat([
    barcelona.reset_index(),
    singapore.reset_index(),
    tokyo.reset_index()
], ignore_index=True)

print(f"Total street segments across all cities:")
print(f"Barcelona: {len(barcelona)}")
print(f"Singapore: {len(singapore)}")
print(f"Tokyo: {len(tokyo)}")
print(f"Combined (3 cities): {len(all_cities)}")
print(f"\nColumns available: {list(all_cities.columns)}")

In [ ]:
# ── DEFINE FEATURE COLUMNS ─────────────────────────────────
feature_cols = [
    'length_m',
    'building_count_50m',
    'avg_building_area',
    'poi_count_50m',
    'highway_rank',
    'sinuosity',
    'poi_diversity',
    'food_drink_count',
    'street_furniture_count',
    'avg_connectivity'
]

# ── NORMALISE AND CLUSTER EACH CITY ────────────────────────
# Using k=5 to match London pilot

cluster_names = {
    0: 'Active Commercial High Street',
    1: 'Arterial Movement Corridor',
    2: 'Local Mixed Street',
    3: 'Institutional Dead-Frontage Street',
    4: 'Winding Historic Lane'
}

colours = {
    'Active Commercial High Street': '#BA7517',
    'Arterial Movement Corridor': '#378ADD',
    'Local Mixed Street': '#888780',
    'Institutional Dead-Frontage Street': '#D85A30',
    'Winding Historic Lane': '#1D9E75'
}

def cluster_city(gdf, city_name):
    X = gdf[feature_cols].fillna(0).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
    gdf = gdf.copy()
    gdf['cluster'] = kmeans.fit_predict(X_scaled)
    
    # Show cluster profiles to help name them
    print(f"\n{city_name} cluster profiles:")
    print(gdf.groupby('cluster')[feature_cols].mean().round(2))
    print(f"\n{city_name} cluster sizes:")
    print(gdf['cluster'].value_counts().sort_index())
    
    return gdf, scaler, kmeans

barcelona_clustered, bcn_scaler, bcn_kmeans = cluster_city(barcelona.reset_index(), "Barcelona")
singapore_clustered, sgp_scaler, sgp_kmeans = cluster_city(singapore.reset_index(), "Singapore")
tokyo_clustered, tky_scaler, tky_kmeans = cluster_city(tokyo.reset_index(), "Tokyo")

print("\nClustering complete for all three cities.")

In [ ]:
# ── ASSIGN TYPOLOGY NAMES PER CITY ─────────────────────────

barcelona_names = {
    0: 'Institutional Large-Block Street',
    1: 'Active Commercial High Street',
    2: 'Local Mixed Street',
    3: 'Arterial Movement Corridor',
    4: 'Active Boulevard'
}

singapore_names = {
    0: 'Arterial Movement Corridor',
    1: 'Active Commercial High Street',
    2: 'Local Mixed Street',
    3: 'Winding Connector Lane',
    4: 'Pedestrian-Rich Street'
}

tokyo_names = {
    0: 'Local Mixed Street',
    1: 'Dense Local Street',
    2: 'Winding Historic Lane',
    3: 'Local Mixed Street',
    4: 'Active Commercial High Street'
}

barcelona_clustered['typology'] = barcelona_clustered['cluster'].map(barcelona_names)
singapore_clustered['typology'] = singapore_clustered['cluster'].map(singapore_names)
tokyo_clustered['typology'] = tokyo_clustered['cluster'].map(tokyo_names)

print("Barcelona typologies:")
print(barcelona_clustered['typology'].value_counts())
print("\nSingapore typologies:")
print(singapore_clustered['typology'].value_counts())
print("\nTokyo typologies:")
print(tokyo_clustered['typology'].value_counts())

In [ ]:
# ── TYPOLOGY MAPS FOR ALL THREE CITIES ────────────────────

colours = {
    'Active Commercial High Street': '#BA7517',
    'Arterial Movement Corridor': '#378ADD',
    'Local Mixed Street': '#888780',
    'Institutional Large-Block Street': '#D85A30',
    'Winding Historic Lane': '#1D9E75',
    'Active Boulevard': '#9B59B6',
    'Dense Local Street': '#2C3E50',
    'Winding Connector Lane': '#27AE60',
    'Pedestrian-Rich Street': '#E74C3C'
}

cities_data = {
    'Barcelona': barcelona_clustered,
    'Singapore': singapore_clustered,
    'Tokyo': tokyo_clustered
}

for city_name, gdf in cities_data.items():
    fig, ax = plt.subplots(figsize=(14, 10))
    
    for typology in gdf['typology'].unique():
        subset = gdf[gdf['typology'] == typology]
        colour = colours.get(typology, '#CCCCCC')
        subset.plot(ax=ax, color=colour, linewidth=1.5,
                   label=f'{typology} (n={len(subset)})',
                   alpha=0.8)
    
    ax.set_title(f"{city_name} — Street Typology Classification\n"
                 f"(Movement and Place framework, TfL 2017)",
                 fontsize=14)
    ax.set_axis_off()
    ax.legend(loc='lower left', fontsize=9, framealpha=0.9)
    
    filename = city_name.lower().replace(' ', '_')
    plt.savefig(f"/home/jovyan/work/outputs/{filename}_typology_map.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"{city_name} map saved.")

In [ ]:
# ── CROSS-CITY TYPOLOGY COMPARISON TABLE ──────────────────

london_typologies = pd.read_csv("/home/jovyan/work/data/london_final_features.csv")

summary = []

city_dfs = {
    'London': london_typologies,
    'Barcelona': barcelona_clustered,
    'Singapore': singapore_clustered,
    'Tokyo': tokyo_clustered
}

for city, df in city_dfs.items():
    total = len(df)
    counts = df['typology'].value_counts()
    for typology, count in counts.items():
        summary.append({
            'City': city,
            'Typology': typology,
            'Count': count,
            'Percentage': round(count / total * 100, 1)
        })

summary_df = pd.DataFrame(summary)
pivot = summary_df.pivot_table(
    index='Typology',
    columns='City',
    values='Percentage',
    aggfunc='sum'
).fillna(0).round(1)

print("Cross-city typology distribution (% of segments):")
print(pivot)

pivot.to_csv("/home/jovyan/work/data/cross_city_comparison.csv")
print("\nSaved to cross_city_comparison.csv")

In [ ]:
# ── STANDARDISE LONDON TYPOLOGY NAMES ─────────────────────
# London was clustered separately so names need aligning
# with the other three cities for cross-city comparison

london_name_map = {
    'Active Commercial Core': 'Active Commercial High Street',
    'Moderate Mixed Street': 'Local Mixed Street',
    'Institutional Large-Block Street': 'Institutional Large-Block Street',
    'Winding Historic Lane': 'Winding Historic Lane',
    'Pedestrian-Rich Street': 'Pedestrian-Rich Street'
}

london_typologies['typology'] = london_typologies['typology'].map(
    london_name_map)

print("London typologies after standardisation:")
print(london_typologies['typology'].value_counts())

In [ ]:
# ── FIX TOKYO — rename cluster 0 to differentiate ─────────

tokyo_name_fix = {
    'Dense Local Street': 'Dense Local Street',
    'Local Mixed Street': 'Local Mixed Street',
    'Active Commercial High Street': 'Active Commercial High Street',
    'Winding Historic Lane': 'Winding Historic Lane'
}

# Cluster 0 had higher highway rank and longer segments
# Rename those specifically
tokyo_clustered.loc[
    tokyo_clustered['cluster'] == 0, 'typology'
] = 'Arterial Movement Corridor'

print("Tokyo typologies after fix:")
print(tokyo_clustered['typology'].value_counts())

In [ ]:
# ── SAVE ALL RESULTS ───────────────────────────────────────

barcelona_clustered.to_file(
    "/home/jovyan/work/data/barcelona_final.gpkg", driver="GPKG")
singapore_clustered.to_file(
    "/home/jovyan/work/data/singapore_final.gpkg", driver="GPKG")
tokyo_clustered.to_file(
    "/home/jovyan/work/data/tokyo_final.gpkg", driver="GPKG")

print("All results saved.")
print("\nSummary of completed analysis:")
print(f"London:    {len(london_typologies)} segments, "
      f"{london_typologies['typology'].nunique()} typologies")
print(f"Barcelona: {len(barcelona_clustered)} segments, "
      f"{barcelona_clustered['typology'].nunique()} typologies")
print(f"Singapore: {len(singapore_clustered)} segments, "
      f"{singapore_clustered['typology'].nunique()} typologies")
print(f"Tokyo:     {len(tokyo_clustered)} segments, "
      f"{tokyo_clustered['typology'].nunique()} typologies")
print(f"\nTotal segments analysed: "
      f"{len(london_typologies) + len(barcelona_clustered) + len(singapore_clustered) + len(tokyo_clustered)}")

In [ ]:
# ── FINAL CROSS-CITY COMPARISON TABLE ─────────────────────

summary = []

city_dfs = {
    'London': london_typologies,
    'Barcelona': barcelona_clustered,
    'Singapore': singapore_clustered,
    'Tokyo': tokyo_clustered
}

for city, df in city_dfs.items():
    total = len(df)
    counts = df['typology'].value_counts()
    for typology, count in counts.items():
        summary.append({
            'City': city,
            'Typology': typology,
            'Count': count,
            'Percentage': round(count / total * 100, 1)
        })

summary_df = pd.DataFrame(summary)
pivot = summary_df.pivot_table(
    index='Typology',
    columns='City',
    values='Percentage',
    aggfunc='sum'
).fillna(0).round(1)

print("Final cross-city typology distribution (% of segments):")
print(pivot)

pivot.to_csv("/home/jovyan/work/data/cross_city_comparison_final.csv")
print("\nSaved to cross_city_comparison_final.csv")

In [ ]:
# ── SIMILARITY SEARCH SYSTEM ───────────────────────────────
# Combines all four cities into one searchable database
# Designer inputs feature values describing a desired street
# System returns the most similar real streets from the database

from sklearn.neighbors import NearestNeighbors

# Combine all four cities into one database
london_geo = london_typologies.copy()
london_geo['city'] = 'London'

barcelona_r = barcelona_clustered.reset_index()
singapore_r = singapore_clustered.reset_index()
tokyo_r = tokyo_clustered.reset_index()

barcelona_r['city'] = 'Barcelona'
singapore_r['city'] = 'Singapore'
tokyo_r['city'] = 'Tokyo'

all_streets = pd.concat([
    london_geo,
    barcelona_r,
    singapore_r,
    tokyo_r
], ignore_index=True)

print(f"Total streets in database: {len(all_streets)}")
print(f"Cities: {all_streets['city'].value_counts().to_dict()}")

# Normalise all features together using one shared scaler
X_all = all_streets[feature_cols].fillna(0).values
scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

# Build KNN model with cosine similarity
knn = NearestNeighbors(n_neighbors=6, metric='cosine')
knn.fit(X_all_scaled)

print("\nSimilarity search system ready.")
print(f"Database: {len(all_streets)} streets across 4 cities.")

In [ ]:
# ── WORKED EXAMPLE 1: Active Commercial High Street ────────
# Designer wants a busy commercial street with lots of
# cafes, shops and pedestrian activity

def search_streets(query_features, label, n=5):
    print(f"\n{'='*55}")
    print(f"Query: {label}")
    print(f"{'='*55}")
    print("Input features:")
    for feat, val in zip(feature_cols, query_features):
        print(f"  {feat}: {val}")
    
    # Normalise query using same scaler
    query = np.array(query_features).reshape(1, -1)
    query_scaled = scaler_all.transform(query)
    
    # Find nearest neighbours
    distances, indices = knn.kneighbors(query_scaled)
    
    # Return results skipping the first if exact match
    results = []
    for i, idx in enumerate(indices[0][1:n+1]):
        row = all_streets.iloc[idx]
        similarity = round((1 - distances[0][i+1]) * 100, 1)
        results.append({
            'Rank': i + 1,
            'Street Name': row.get('name', 'Unnamed'),
            'City': row['city'],
            'Typology': row['typology'],
            'Similarity': f"{similarity}%"
        })
    
    results_df = pd.DataFrame(results)
    print("\nTop 5 most similar streets:")
    print(results_df.to_string(index=False))
    return results_df

# Example 1: Designer wants an active commercial street
# High POI, high food/drink, moderate movement
example1 = search_streets(
    query_features=[
        60,    # length_m — moderate
        25,    # building_count_50m — moderate
        800,   # avg_building_area — small active units
        35,    # poi_count_50m — high
        3,     # highway_rank — tertiary
        1.0,   # sinuosity — straight
        1.4,   # poi_diversity — high mix
        12,    # food_drink_count — high
        4,     # street_furniture_count — moderate
        4.5    # avg_connectivity — moderate
    ],
    label="Active Commercial High Street"
)

In [ ]:
# ── FIX: rebuild all_streets with name column ──────────────

# Add empty name column to London since it was lost
london_geo['name'] = 'Unknown'

# Rebuild the combined database with name column
all_streets = pd.concat([
    london_geo[feature_cols + ['typology', 'city', 'name']],
    barcelona_r[feature_cols + ['typology', 'city', 'name']],
    singapore_r[feature_cols + ['typology', 'city', 'name']],
    tokyo_r[feature_cols + ['typology', 'city', 'name']]
], ignore_index=True)

# Fill any remaining NaN names
all_streets['name'] = all_streets['name'].fillna('Unnamed Street')

# Renormalise and rebuild KNN on the clean database
X_all = all_streets[feature_cols].fillna(0).values
scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

knn = NearestNeighbors(n_neighbors=6, metric='cosine')
knn.fit(X_all_scaled)

print(f"Database rebuilt: {len(all_streets)} streets")
print(f"Name column sample from Barcelona:")
print(all_streets[all_streets['city']=='Barcelona']['name'].dropna().head(5).values)

In [ ]:
# ── WORKED EXAMPLE 1: Active Commercial High Street ────────

def search_streets(query_features, label, n=5):
    print(f"\n{'='*55}")
    print(f"Query: {label}")
    print(f"{'='*55}")

    query = np.array(query_features).reshape(1, -1)
    query_scaled = scaler_all.transform(query)

    distances, indices = knn.kneighbors(query_scaled)

    results = []
    for i, idx in enumerate(indices[0][1:n+1]):
        row = all_streets.iloc[idx]
        similarity = round((1 - distances[0][i+1]) * 100, 1)
        results.append({
            'Rank': i + 1,
            'Street Name': row['name'],
            'City': row['city'],
            'Typology': row['typology'],
            'Similarity': f"{similarity}%"
        })

    results_df = pd.DataFrame(results)
    print("\nTop 5 most similar streets:")
    print(results_df.to_string(index=False))
    return results_df

# Example 1 — designer wants a busy commercial street
# High POI, high food and drink, moderate movement
example1 = search_streets(
    query_features=[
        60,    # length_m
        25,    # building_count_50m
        800,   # avg_building_area
        35,    # poi_count_50m — high
        3,     # highway_rank
        1.0,   # sinuosity — straight
        1.4,   # poi_diversity — high mix
        12,    # food_drink_count — high
        4,     # street_furniture_count
        4.5    # avg_connectivity
    ],
    label="Active Commercial High Street — busy shopping street"
)

# Example 2 — designer wants a quiet historic lane
# Low POI, high sinuosity, low highway rank
example2 = search_streets(
    query_features=[
        40,    # length_m — short
        10,    # building_count_50m
        1200,  # avg_building_area
        5,     # poi_count_50m — low
        1,     # highway_rank — minor
        2.5,   # sinuosity — winding
        0.5,   # poi_diversity — low mix
        1,     # food_drink_count — very low
        2,     # street_furniture_count
        3.0    # avg_connectivity — low
    ],
    label="Winding Historic Lane — quiet curved street"
)

# Example 3 — designer wants a major arterial road
# High highway rank, high betweenness, low POI
example3 = search_streets(
    query_features=[
        120,   # length_m — long
        15,    # building_count_50m
        2000,  # avg_building_area — large buildings
        8,     # poi_count_50m — low
        5,     # highway_rank — primary
        1.0,   # sinuosity — straight
        0.8,   # poi_diversity — low
        2,     # food_drink_count — low
        2,     # street_furniture_count — low
        5.5    # avg_connectivity — high
    ],
    label="Arterial Movement Corridor — major through road"
)

In [ ]:
# ── FIX LONDON STREET NAMES ────────────────────────────────
# Load original London streets which have the name column
london_streets_geo = gpd.read_file(
    "/home/jovyan/work/data/london_streets.gpkg")

print("London streets columns:")
print(london_streets_geo.columns.tolist())
print("\nSample names:")
print(london_streets_geo['name'].dropna().head(10).values)

In [ ]:
# ── MERGE LONDON NAMES INTO FEATURES ──────────────────────

# Load London final features CSV
london_features_csv = pd.read_csv(
    "/home/jovyan/work/data/london_final_features.csv")

# Merge name from the gpkg using u, v, key as the join keys
london_with_names = london_features_csv.merge(
    london_streets_geo[['u', 'v', 'key', 'name']],
    on=['u', 'v', 'key'],
    how='left'
)

# Standardise typology names to match other cities
london_name_map = {
    'Active Commercial Core': 'Active Commercial High Street',
    'Moderate Mixed Street': 'Local Mixed Street',
    'Institutional Large-Block Street': 'Institutional Large-Block Street',
    'Winding Historic Lane': 'Winding Historic Lane',
    'Pedestrian-Rich Street': 'Pedestrian-Rich Street'
}

london_with_names['typology'] = london_with_names['typology'].map(
    london_name_map)
london_with_names['city'] = 'London'
london_with_names['name'] = london_with_names['name'].fillna('Unnamed Street')

print(f"London segments: {len(london_with_names)}")
print(f"Named streets: {london_with_names['name'].ne('Unnamed Street').sum()}")
print(f"\nSample named streets:")
print(london_with_names[['name', 'typology']].head(10))

In [ ]:
# ── REBUILD DATABASE WITH LONDON NAMES ────────────────────

all_streets = pd.concat([
    london_with_names[feature_cols + ['typology', 'city', 'name']],
    barcelona_r[feature_cols + ['typology', 'city', 'name']],
    singapore_r[feature_cols + ['typology', 'city', 'name']],
    tokyo_r[feature_cols + ['typology', 'city', 'name']]
], ignore_index=True)

all_streets['name'] = all_streets['name'].fillna('Unnamed Street')

# Renormalise and rebuild KNN
X_all = all_streets[feature_cols].fillna(0).values
scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

knn = NearestNeighbors(n_neighbors=6, metric='cosine')
knn.fit(X_all_scaled)

print(f"Database rebuilt: {len(all_streets)} streets")
print(f"London named: {(all_streets[all_streets['city']=='London']['name'] != 'Unnamed Street').sum()}")
print(f"Barcelona named: {(all_streets[all_streets['city']=='Barcelona']['name'] != 'Unnamed Street').sum()}")
print(f"Singapore named: {(all_streets[all_streets['city']=='Singapore']['name'] != 'Unnamed Street').sum()}")
print(f"Tokyo named: {(all_streets[all_streets['city']=='Tokyo']['name'] != 'Unnamed Street').sum()}")

In [ ]:
# ── FINAL SIMILARITY SEARCH — ALL THREE EXAMPLES ──────────

def search_streets(query_features, label, n=5):
    print(f"\n{'='*55}")
    print(f"Query: {label}")
    print(f"{'='*55}")

    query = np.array(query_features).reshape(1, -1)
    query_scaled = scaler_all.transform(query)
    distances, indices = knn.kneighbors(query_scaled)

    results = []
    for i, idx in enumerate(indices[0][1:n+1]):
        row = all_streets.iloc[idx]
        similarity = round((1 - distances[0][i+1]) * 100, 1)
        results.append({
            'Rank': i + 1,
            'Street Name': row['name'],
            'City': row['city'],
            'Typology': row['typology'],
            'Similarity': f"{similarity}%"
        })

    results_df = pd.DataFrame(results)
    print("\nTop 5 most similar streets:")
    print(results_df.to_string(index=False))
    return results_df

# Example 1 — Active Commercial High Street
example1 = search_streets(
    query_features=[
        60, 25, 800, 35, 3, 1.0, 1.4, 12, 4, 4.5
    ],
    label="Active Commercial High Street"
)

# Example 2 — Winding Historic Lane
example2 = search_streets(
    query_features=[
        40, 10, 1200, 5, 1, 2.5, 0.5, 1, 2, 3.0
    ],
    label="Winding Historic Lane"
)

# Example 3 — Arterial Movement Corridor
example3 = search_streets(
    query_features=[
        120, 15, 2000, 8, 5, 1.0, 0.8, 2, 2, 5.5
    ],
    label="Arterial Movement Corridor"
)

In [3]:
# ── SAVE FINAL SIMILARITY SEARCH RESULTS ──────────────────

example1.to_csv(
    "/home/jovyan/work/outputs/similarity_example1_final.csv", 
    index=False)
example2.to_csv(
    "/home/jovyan/work/outputs/similarity_example2_final.csv", 
    index=False)
example3.to_csv(
    "/home/jovyan/work/outputs/similarity_example3_final.csv", 
    index=False)

print("Similarity search results saved.")
print("\nFull analysis complete.")
print(f"\nWhat you have built:")
print(f"  4 cities analysed: London, Barcelona, Singapore, Tokyo")
print(f"  5,010 street segments in database")
print(f"  10 features per segment")
print(f"  5 typologies per city")
print(f"  3 worked similarity search examples")
print(f"  All maps, tables and results saved")

NameError: name 'example1' is not defined

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Predefined query profiles for each typology
typology_queries = {
    'Active Commercial High Street': [60, 25, 800, 35, 3, 1.0, 1.4, 12, 4, 4.5],
    'Arterial Movement Corridor':    [120, 15, 2000, 8, 5, 1.0, 0.8, 2, 2, 5.5],
    'Local Mixed Street':            [55, 16, 1200, 15, 2, 1.0, 1.1, 4, 3, 4.5],
    'Institutional Large-Block Street': [40, 8, 2500, 6, 2, 1.0, 0.7, 1, 1, 4.0],
    'Winding Historic Lane':         [40, 10, 1200, 5, 1, 2.5, 0.5, 1, 2, 3.0]
}

# Dropdown
typology_dropdown = widgets.Dropdown(
    options=list(typology_queries.keys()),
    description='Street type:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

# Button
search_button = widgets.Button(
    description='Find Similar Streets',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

# Output area
output = widgets.Output()

def run_search(button):
    with output:
        clear_output()
        
        selected = typology_dropdown.value
        query_features = typology_queries[selected]
        
        print(f"Searching for streets similar to: {selected}")
        print("-" * 55)
        
        query = np.array(query_features).reshape(1, -1)
        query_scaled = scaler_all.transform(query)
        distances, indices = knn.kneighbors(query_scaled)
        
        results = []
        for i, idx in enumerate(indices[0][1:6]):
            row = all_streets.iloc[idx]
            similarity = round((1 - distances[0][i+1]) * 100, 1)
            results.append({
                'Rank': i + 1,
                'Street Name': row['name'],
                'City': row['city'],
                'Typology': row['typology'],
                'Similarity': f"{similarity}%"
            })
        
        results_df = pd.DataFrame(results)
        display(results_df)

search_button.on_click(run_search)

print("Street Typology Explorer")
print("Select a street type and click Search")
display(typology_dropdown)
display(search_button)
display(output)

In [ ]:
# ── CLEAN STREET NAMES WITH MULTIPLE VALUES ────────────────

def clean_name(name):
    if isinstance(name, list):
        return name[0]
    if str(name).startswith('['):
        # String representation of a list
        return str(name).strip("[]'\"").split("',")[0].strip("' ")
    return name

all_streets['name'] = all_streets['name'].apply(clean_name)

# Rebuild KNN with cleaned names
X_all = all_streets[feature_cols].fillna(0).values
scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)
knn = NearestNeighbors(n_neighbors=6, metric='cosine')
knn.fit(X_all_scaled)

print("Names cleaned.")
print("Sample check:")
print(all_streets['name'].head(20).values)

In [ ]:
# ── QUICK TEST ─────────────────────────────────────────────

query_features = typology_queries['Winding Historic Lane']
query = np.array(query_features).reshape(1, -1)
query_scaled = scaler_all.transform(query)
distances, indices = knn.kneighbors(query_scaled)

results = []
for i, idx in enumerate(indices[0][1:6]):
    row = all_streets.iloc[idx]
    similarity = round((1 - distances[0][i+1]) * 100, 1)
    results.append({
        'Rank': i + 1,
        'Street Name': row['name'],
        'City': row['city'],
        'Typology': row['typology'],
        'Similarity': f"{similarity}%"
    })

print(pd.DataFrame(results).to_string(index=False))

In [ ]:
# ── IMPROVED TYPOLOGY MAPS WITH BASEMAP ────────────────────
import contextily as ctx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

def plot_typology_map_with_basemap(gdf, city_name, colours, filename):
    # Convert to Web Mercator for basemap
    gdf_web = gdf.to_crs(epsg=3857)
    
    fig, ax = plt.subplots(figsize=(14, 12))
    
    plotted_typologies = []
    for typology, colour in colours.items():
        subset = gdf_web[gdf_web['typology'] == typology]
        if len(subset) > 0:
            subset.plot(
                ax=ax,
                color=colour,
                linewidth=2.5,
                alpha=0.9,
                zorder=2
            )
            plotted_typologies.append(
                mpatches.Patch(
                    color=colour,
                    label=f'{typology} (n={len(subset)})'
                )
            )
    
    # Add basemap
    ctx.add_basemap(
        ax,
        source=ctx.providers.CartoDB.Positron,
        zoom='auto'
    )
    
    ax.set_title(
        f"{city_name} — Street Typology Classification\n"
        f"Movement and Place Framework (Transport for London, 2017)",
        fontsize=16, fontweight='bold', pad=15
    )
    ax.set_axis_off()
    ax.legend(
        handles=plotted_typologies,
        loc='lower left',
        fontsize=10,
        framealpha=0.95,
        edgecolor='#CCCCCC',
        fancybox=True
    )
    
    plt.tight_layout()
    plt.savefig(
        f"/home/jovyan/work/outputs/{filename}",
        dpi=200,
        bbox_inches='tight',
        facecolor='white'
    )
    plt.show()
    print(f"{city_name} map saved.")

# Plot London
london_gdf = gpd.read_file("/home/jovyan/work/data/london_streets.gpkg")
london_features = pd.read_csv("/home/jovyan/work/data/london_final_features.csv")
london_features['typology'] = london_features['typology'].map({
    'Active Commercial Core': 'Active Commercial High Street',
    'Moderate Mixed Street': 'Local Mixed Street',
    'Institutional Large-Block Street': 'Institutional Large-Block Street',
    'Winding Historic Lane': 'Winding Historic Lane',
    'Pedestrian-Rich Street': 'Pedestrian-Rich Street'
})
london_merged = london_gdf.merge(
    london_features[['u', 'v', 'key', 'typology']],
    on=['u', 'v', 'key'], how='left'
)

colours = {
    'Active Commercial High Street': '#BA7517',
    'Arterial Movement Corridor': '#378ADD',
    'Local Mixed Street': '#888780',
    'Institutional Large-Block Street': '#D85A30',
    'Winding Historic Lane': '#1D9E75',
    'Pedestrian-Rich Street': '#9B59B6'
}

plot_typology_map_with_basemap(
    london_merged,
    'City of London',
    colours,
    'london_typology_basemap.png'
)

In [ ]:
# ── BASEMAP MAPS FOR ALL THREE CITIES ─────────────────────

# Barcelona
plot_typology_map_with_basemap(
    barcelona_clustered.to_crs(epsg=3857),
    'Barcelona — Eixample',
    colours,
    'barcelona_typology_basemap.png'
)

# Singapore
plot_typology_map_with_basemap(
    singapore_clustered.to_crs(epsg=3857),
    'Singapore — Downtown Core',
    colours,
    'singapore_typology_basemap.png'
)

# Tokyo
plot_typology_map_with_basemap(
    tokyo_clustered.to_crs(epsg=3857),
    'Tokyo — Shinjuku',
    colours,
    'tokyo_typology_basemap.png'
)

In [ ]:
# ── RADAR CHARTS PER TYPOLOGY ──────────────────────────────
from matplotlib.patches import FancyArrowPatch
import matplotlib.gridspec as gridspec

# Use London data for radar charts
london_features_radar = pd.read_csv(
    "/home/jovyan/work/data/london_final_features.csv")
london_features_radar['typology'] = london_features_radar['typology'].map({
    'Active Commercial Core': 'Active Commercial High Street',
    'Moderate Mixed Street': 'Local Mixed Street',
    'Institutional Large-Block Street': 'Institutional Large-Block Street',
    'Winding Historic Lane': 'Winding Historic Lane',
    'Pedestrian-Rich Street': 'Pedestrian-Rich Street'
})

feature_cols = [
    'length_m', 'building_count_50m', 'avg_building_area',
    'poi_count_50m', 'highway_rank', 'sinuosity',
    'poi_diversity', 'food_drink_count',
    'street_furniture_count', 'avg_connectivity'
]

feature_labels = [
    'Length', 'Buildings', 'Bldg Area',
    'POI Count', 'Hwy Rank', 'Sinuosity',
    'POI Diversity', 'Food/Drink',
    'Street Furn.', 'Connectivity'
]

colours = {
    'Active Commercial High Street': '#BA7517',
    'Arterial Movement Corridor': '#378ADD',
    'Local Mixed Street': '#888780',
    'Institutional Large-Block Street': '#D85A30',
    'Winding Historic Lane': '#1D9E75',
    'Pedestrian-Rich Street': '#9B59B6'
}

# Normalise features to 0-1 for radar
from sklearn.preprocessing import MinMaxScaler
scaler_radar = MinMaxScaler()
london_features_radar[feature_cols] = scaler_radar.fit_transform(
    london_features_radar[feature_cols].fillna(0))

# Calculate mean per typology
profiles = london_features_radar.groupby('typology')[feature_cols].mean()

# Number of features
N = len(feature_cols)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

typologies = [t for t in colours.keys() if t in profiles.index]

fig, axes = plt.subplots(
    2, 3, figsize=(18, 12),
    subplot_kw=dict(polar=True)
)
axes = axes.flatten()

for idx, typology in enumerate(typologies):
    ax = axes[idx]
    values = profiles.loc[typology].values.tolist()
    values += values[:1]
    
    colour = colours[typology]
    
    ax.plot(angles, values, 'o-', linewidth=2, color=colour)
    ax.fill(angles, values, alpha=0.25, color=colour)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(feature_labels, size=9)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.5, 0.75])
    ax.set_yticklabels(['0.25', '0.50', '0.75'], size=7, color='grey')
    ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_title(
        typology,
        size=12, fontweight='bold',
        color=colour, pad=15
    )
    ax.spines['polar'].set_color(colour)
    ax.spines['polar'].set_linewidth(1.5)

# Hide unused subplot
if len(typologies) < 6:
    for i in range(len(typologies), 6):
        axes[i].set_visible(False)

fig.suptitle(
    'Street Typology Feature Profiles — City of London\n'
    'Normalised mean values across 10 spatial features',
    fontsize=16, fontweight='bold', y=1.01
)

plt.tight_layout()
plt.savefig(
    "/home/jovyan/work/outputs/radar_charts.png",
    dpi=200, bbox_inches='tight', facecolor='white'
)
plt.show()
print("Radar charts saved.")

In [ ]:
# ── RELOAD LONDON RADAR DATA ───────────────────────────────
london_radar = pd.read_csv(
    "/home/jovyan/work/data/london_final_features.csv")
london_radar['typology'] = london_radar['typology'].map({
    'Active Commercial Core': 'Active Commercial High Street',
    'Moderate Mixed Street': 'Local Mixed Street',
    'Institutional Large-Block Street': 'Institutional Large-Block Street',
    'Winding Historic Lane': 'Winding Historic Lane',
    'Pedestrian-Rich Street': 'Pedestrian-Rich Street'
})
london_radar['city'] = 'London'

barcelona_radar = barcelona_clustered.reset_index().copy()
barcelona_radar['city'] = 'Barcelona'
singapore_radar = singapore_clustered.reset_index().copy()
singapore_radar['city'] = 'Singapore'
tokyo_radar = tokyo_clustered.reset_index().copy()
tokyo_radar['city'] = 'Tokyo'

print("All city radar dataframes ready.")
print(f"London: {len(london_radar)} segments")
print(f"Barcelona: {len(barcelona_radar)} segments")
print(f"Singapore: {len(singapore_radar)} segments")
print(f"Tokyo: {len(tokyo_radar)} segments")

In [ ]:
# ── DEFINE RADAR VARIABLES ─────────────────────────────────
from sklearn.preprocessing import MinMaxScaler

feature_cols_radar = [
    'length_m', 'building_count_50m', 'avg_building_area',
    'poi_count_50m', 'highway_rank', 'sinuosity',
    'poi_diversity', 'food_drink_count',
    'street_furniture_count', 'avg_connectivity'
]

feature_labels_radar = [
    'Length', 'Buildings', 'Bldg Area',
    'POI Count', 'Hwy Rank', 'Sinuosity',
    'POI Diversity', 'Food/Drink',
    'Street Furn.', 'Connectivity'
]

N = len(feature_cols_radar)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

colours = {
    'Active Commercial High Street': '#BA7517',
    'Arterial Movement Corridor': '#378ADD',
    'Local Mixed Street': '#888780',
    'Institutional Large-Block Street': '#D85A30',
    'Winding Historic Lane': '#1D9E75',
    'Pedestrian-Rich Street': '#9B59B6',
    'Dense Local Street': '#2C3E50',
    'Active Boulevard': '#8E44AD',
    'Winding Connector Lane': '#27AE60',
}

print("Radar variables defined.")

In [ ]:
# ── ALIASES TO FIX VARIABLE NAME CONFLICT ─────────────────
feature_cols_radar = feature_cols
feature_labels_radar = feature_labels

# ── COMBINED RADAR CHART — KEY TYPOLOGIES ACROSS CITIES ───

key_typologies = [
    'Active Commercial High Street',
    'Local Mixed Street',
    'Arterial Movement Corridor'
]

city_names = ['London', 'Barcelona', 'Singapore', 'Tokyo']
city_dfs = {
    'London': london_radar,
    'Barcelona': barcelona_radar,
    'Singapore': singapore_radar,
    'Tokyo': tokyo_radar
}

fig, axes = plt.subplots(
    len(key_typologies), len(city_names),
    figsize=(20, 14),
    subplot_kw=dict(polar=True)
)

for row_idx, typology in enumerate(key_typologies):
    for col_idx, city_name in enumerate(city_names):
        ax = axes[row_idx][col_idx]
        city_df = city_dfs[city_name].copy()

        scaler_c = MinMaxScaler()
        city_df[feature_cols_radar] = scaler_c.fit_transform(
            city_df[feature_cols_radar].fillna(0))

        profiles = city_df.groupby('typology')[feature_cols_radar].mean()
        colour = colours.get(typology, '#888780')

        if typology in profiles.index:
            values = profiles.loc[typology].values.tolist()
            values += values[:1]
            ax.plot(angles, values, 'o-', linewidth=2, color=colour)
            ax.fill(angles, values, alpha=0.25, color=colour)
        else:
            ax.text(0, 0, 'Not present\nin this city',
                   ha='center', va='center', fontsize=9,
                   color='#AAAAAA')

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(feature_labels_radar, size=7)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.25, 0.5, 0.75])
        ax.set_yticklabels([])
        ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
        ax.spines['polar'].set_color(colour)

        if row_idx == 0:
            ax.set_title(city_name, size=13, fontweight='bold',
                        pad=20, color='#1A1916')

        if col_idx == 0:
            ax.set_ylabel(typology, size=10, fontweight='bold',
                         color=colour, labelpad=30)

fig.suptitle(
    'Key Street Typology Feature Profiles Across Four Cities\n'
    'Normalised mean values across 10 spatial features',
    fontsize=15, fontweight='bold', y=1.01
)

plt.tight_layout()
plt.savefig(
    "/home/jovyan/work/outputs/radar_comparison.png",
    dpi=200, bbox_inches='tight', facecolor='white'
)
plt.show()
print("Combined radar comparison saved.")

In [ ]:
# ── BOX PLOTS — HUMAN-CENTRED FEATURES BY TYPOLOGY ────────
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

human_features = [
    ('food_drink_count', 'Food and Drink Count', 
     'Active frontage proxy (Gehl, 2010)'),
    ('street_furniture_count', 'Street Furniture Count',
     'Pedestrian amenity (Gehl, 2010)'),
    ('poi_diversity', 'POI Diversity (Shannon Entropy)',
     'Land use mix (Jacobs, 1993)')
]

# Use raw unnormalised London data
london_raw = pd.read_csv(
    "/home/jovyan/work/data/london_final_features.csv")
london_raw['typology'] = london_raw['typology'].map({
    'Active Commercial Core': 'Active Commercial High Street',
    'Moderate Mixed Street': 'Local Mixed Street',
    'Institutional Large-Block Street': 'Institutional Large-Block Street',
    'Winding Historic Lane': 'Winding Historic Lane',
    'Pedestrian-Rich Street': 'Pedestrian-Rich Street'
})

typology_order = [
    'Active Commercial High Street',
    'Pedestrian-Rich Street',
    'Local Mixed Street',
    'Institutional Large-Block Street',
    'Winding Historic Lane'
]

palette = {
    'Active Commercial High Street': '#BA7517',
    'Pedestrian-Rich Street': '#9B59B6',
    'Local Mixed Street': '#888780',
    'Institutional Large-Block Street': '#D85A30',
    'Winding Historic Lane': '#1D9E75'
}

for idx, (feature, label, subtitle) in enumerate(human_features):
    ax = axes[idx]
    
    sns.boxplot(
        data=london_raw,
        x='typology',
        y=feature,
        order=typology_order,
        palette=palette,
        ax=ax,
        width=0.6,
        flierprops=dict(marker='o', markersize=3, alpha=0.4),
        linewidth=1.2
    )
    
    ax.set_title(
        f'{label}\n',
        fontsize=13, fontweight='bold'
    )
    ax.set_xlabel('')
    ax.set_ylabel(label, fontsize=11)
    ax.set_xticklabels(
        [t.replace(' ', '\n') for t in typology_order],
        fontsize=8, rotation=0
    )
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    ax.text(
        0.5, -0.18, subtitle,
        transform=ax.transAxes,
        ha='center', fontsize=9,
        style='italic', color='#666666'
    )

fig.suptitle(
    'Human-Centred Street Quality by Typology — City of London\n'
    'Distribution of pedestrian-oriented features across five street types',
    fontsize=15, fontweight='bold', y=1.02
)

plt.tight_layout()
plt.savefig(
    "/home/jovyan/work/outputs/boxplots_human_centred.png",
    dpi=200, bbox_inches='tight', facecolor='white'
)
plt.show()
print("Box plots saved.")

In [ ]:
# ── CORRELATION HEATMAP ────────────────────────────────────
import seaborn as sns

fig, ax = plt.subplots(figsize=(12, 10))

feature_labels_short = {
    'length_m': 'Length',
    'building_count_50m': 'Buildings',
    'avg_building_area': 'Bldg Area',
    'poi_count_50m': 'POI Count',
    'highway_rank': 'Hwy Rank',
    'sinuosity': 'Sinuosity',
    'poi_diversity': 'POI Diversity',
    'food_drink_count': 'Food/Drink',
    'street_furniture_count': 'St. Furniture',
    'avg_connectivity': 'Connectivity'
}

corr_data = london_raw[list(feature_labels_short.keys())].rename(
    columns=feature_labels_short)
corr_matrix = corr_data.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    vmin=-1, vmax=1,
    ax=ax,
    square=True,
    linewidths=0.5,
    linecolor='white',
    annot_kws={'size': 9},
    cbar_kws={'shrink': 0.8, 'label': 'Pearson correlation coefficient'}
)

ax.set_title(
    'Feature Correlation Matrix — City of London\n'
    'Pearson correlation coefficients across 10 spatial features',
    fontsize=14, fontweight='bold', pad=15
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig(
    "/home/jovyan/work/outputs/correlation_heatmap.png",
    dpi=200, bbox_inches='tight', facecolor='white'
)
plt.show()
print("Correlation heatmap saved.")

In [ ]:
# ── CROSS-CITY COMPARISON BAR CHART ───────────────────────

comparison = pd.read_csv(
    "/home/jovyan/work/data/cross_city_comparison_final.csv",
    index_col=0
)

# Keep main typologies only
main_typologies = [
    'Active Commercial High Street',
    'Arterial Movement Corridor',
    'Local Mixed Street',
    'Institutional Large-Block Street',
    'Winding Historic Lane'
]

comparison_filtered = comparison.loc[
    comparison.index.isin(main_typologies)
].fillna(0)

cities = ['London', 'Barcelona', 'Singapore', 'Tokyo']
comparison_filtered = comparison_filtered[cities]

fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(cities))
width = 0.15
multiplier = 0

bar_colours = {
    'Active Commercial High Street': '#BA7517',
    'Arterial Movement Corridor': '#378ADD',
    'Local Mixed Street': '#888780',
    'Institutional Large-Block Street': '#D85A30',
    'Winding Historic Lane': '#1D9E75'
}

for typology in main_typologies:
    if typology in comparison_filtered.index:
        values = comparison_filtered.loc[typology, cities].values
        offset = width * multiplier
        bars = ax.bar(
            x + offset, values, width,
            label=typology,
            color=bar_colours[typology],
            alpha=0.9,
            edgecolor='white',
            linewidth=0.5
        )
        for bar, val in zip(bars, values):
            if val > 2:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5,
                    f'{val:.0f}%',
                    ha='center', va='bottom',
                    fontsize=7, color='#444444'
                )
        multiplier += 1

ax.set_xlabel('City', fontsize=12)
ax.set_ylabel('Percentage of street segments (%)', fontsize=12)
ax.set_title(
    'Street Typology Distribution Across Four Global Cities\n'
    'Percentage of segments per typology (K-means, k=5)',
    fontsize=14, fontweight='bold'
)
ax.set_xticks(x + width * 2)
ax.set_xticklabels(cities, fontsize=12)
ax.legend(
    loc='upper right',
    fontsize=9,
    framealpha=0.95,
    edgecolor='#CCCCCC'
)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, 75)

plt.tight_layout()
plt.savefig(
    "/home/jovyan/work/outputs/cross_city_bar_chart.png",
    dpi=200, bbox_inches='tight', facecolor='white'
)
plt.show()
print("Cross-city bar chart saved.")

In [ ]:
from sklearn.metrics import silhouette_score

X_london = london_features_radar[feature_cols].fillna(0).values
scaler_s = StandardScaler()
X_scaled_s = scaler_s.fit_transform(X_london)

silhouette_scores = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled_s)
    score = silhouette_score(X_scaled_s, labels)
    silhouette_scores.append(score)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, [9500, 9000, 8500, 8100, 7900, 7700, 7550, 7420, 7300],
             'o-', color='#378ADD', linewidth=2, markersize=8)
axes[0].set_title('Elbow Method — Inertia', fontweight='bold')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].grid(alpha=0.3)

axes[1].plot(k_range, silhouette_scores, 'o-', 
             color='#BA7517', linewidth=2, markersize=8)
axes[1].axvline(x=5, color='#D85A30', linestyle='--', alpha=0.7, label='k=5 selected')
axes[1].set_title('Silhouette Score', fontweight='bold')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette score')
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.suptitle('K-means Cluster Validation — City of London\n'
             'Elbow method and silhouette score',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig("/home/jovyan/work/outputs/cluster_validation.png",
            dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print(f"Silhouette scores: {[round(s,3) for s in silhouette_scores]}")
print(f"Best k by silhouette: {k_range[silhouette_scores.index(max(silhouette_scores))]}")